# Telecom Customer Churn Analysis
## 01 — Initial Data Audit

### Objective
To understand the structure, quality, and characteristics of the telecom customer dataset before performing analysis.

In [1]:
import pandas as pd

In [2]:
dataset = r"C:\Users\HomePC\Desktop\Data Analysis Portfolio\telecom_customer_churn_analysis\data\telecom_customer_churn.csv"
df = pd.read_csv(dataset)

In [3]:
df.head()

,Customer ID,Gender,Age,Married,Number of Dependents,City,Zip Code,Latitude,Longitude,Number of Referrals,...,Payment Method,Monthly Charge,Total Charges,Total Refunds,Total Extra Data Charges,Total Long Distance Charges,Total Revenue,Customer Status,Churn Category,Churn Reason
0,0002-ORFBO,Female,37,Yes,0,Frazier Park,93225,34.827662,-118.999073,2,...,Credit Card,65.6,593.30,0.00,0,381.51,974.81,Stayed,NaN,NaN
1,0003-MKNFE,Male,46,No,0,Glendale,91206,34.162515,-118.203869,0,...,Credit Card,-4.0,542.40,38.33,10,96.21,610.28,Stayed,NaN,NaN
2,0004-TLHLJ,Male,50,No,0,Costa Mesa,92627,33.645672,-117.922613,0,...,Bank Withdrawal,73.9,280.85,0.00,0,134.60,415.45,Churned,Competitor,Competitor had better devices
3,0011-IGKFF,Male,78,Yes,0,Martinez,94553,38.014457,-122.115432,1,...,Bank Withdrawal,98.0,1237.85,0.00,0,361.66,1599.51,Churned,Dissatisfaction,Product dissatisfaction
4,0013-EXCHZ,Female,75,Yes,0,Camarillo,93010,34.227846,-119.079903,3,...,Credit Card,83.9,267.40,0.00,0,22.14,289.54,Churned,Dissatisfaction,Network reliability


In [4]:
df.shape

(7043, 38)

In [5]:
df.columns.tolist()

['Customer ID',
 'Gender',
 'Age',
 'Married',
 'Number of Dependents',
 'City',
 'Zip Code',
 'Latitude',
 'Longitude',
 'Number of Referrals',
 'Tenure in Months',
 'Offer',
 'Phone Service',
 'Avg Monthly Long Distance Charges',
 'Multiple Lines',
 'Internet Service',
 'Internet Type',
 'Avg Monthly GB Download',
 'Online Security',
 'Online Backup',
 'Device Protection Plan',
 'Premium Tech Support',
 'Streaming TV',
 'Streaming Movies',
 'Streaming Music',
 'Unlimited Data',
 'Contract',
 'Paperless Billing',
 'Payment Method',
 'Monthly Charge',
 'Total Charges',
 'Total Refunds',
 'Total Extra Data Charges',
 'Total Long Distance Charges',
 'Total Revenue',
 'Customer Status',
 'Churn Category',
 'Churn Reason']

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 38 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Customer ID                        7043 non-null   object 
 1   Gender                             7043 non-null   object 
 2   Age                                7043 non-null   int64  
 3   Married                            7043 non-null   object 
 4   Number of Dependents               7043 non-null   int64  
 5   City                               7043 non-null   object 
 6   Zip Code                           7043 non-null   int64  
 7   Latitude                           7043 non-null   float64
 8   Longitude                          7043 non-null   float64
 9   Number of Referrals                7043 non-null   int64  
 10  Tenure in Months                   7043 non-null   int64  
 11  Offer                              3166 non-null   objec

## Observations from the dataset overview

- The dataset contains 7,043 customer records and 38 columns.
- It contains a mixture of categorical and numerical variables.
- Several columns contain missing values.
- Missing values appear to be concentrated in service-related variables and churn-related fields.
- The data dictionary indicates that some service fields may be unavailable when a customer does not subscribe to the corresponding service.
- Churn Category and Churn Reason contain fewer non-null records than the total customer population and require further investigation.
- Further checks are needed to distinguish genuine missing data from structurally inapplicable values.

In [56]:
missing = df.isnull().sum()

missing[missing > 0]

Offer                                3877
Avg Monthly Long Distance Charges     682
Multiple Lines                        682
Internet Type                        1526
Avg Monthly GB Download              1526
Online Security                      1526
Online Backup                        1526
Device Protection Plan               1526
Premium Tech Support                 1526
Streaming TV                         1526
Streaming Movies                     1526
Streaming Music                      1526
Unlimited Data                       1526
Churn Category                       5174
Churn Reason                         5174
dtype: int64

### Checking the Internet Service Columns

In [8]:
df["Internet Service"].value_counts(dropna=False)

Internet Service
Yes    5517
No     1526
Name: count, dtype: int64

In [9]:
pd.crosstab(
    df["Internet Service"],
    df["Internet Type"],
    dropna=False
)

Internet Type,Cable,DSL,Fiber Optic,NaN
Internet Service,,,,
No,0,0,0,1526
Yes,830,1652,3035,0


### Missing Internet Service Fields

The `Internet Type` field contains 1,526 missing values.

A cross-tabulation of `Internet Service` and `Internet Type` shows that all 1,526 missing values belong to customers where `Internet Service = No`.

Customers with `Internet Service = Yes` have a recorded internet type.

Therefore, these missing values appear to represent structurally inapplicable data rather than missing data-entry values. The data dictionary also indicates that `Internet Type` is `None` when a customer does not subscribe to internet service.

No values will be modified during the initial audit.

### Checking the Phone Service Columns

In [10]:
pd.crosstab(
    df["Phone Service"],
    df["Multiple Lines"],
    dropna=False
)

Multiple Lines,No,Yes,NaN
Phone Service,,,
No,0,0,682
Yes,3390,2971,0


In [11]:
df.groupby("Phone Service")["Avg Monthly Long Distance Charges"].apply(
    lambda x: x.isna().sum()
)

Phone Service
No     682
Yes      0
Name: Avg Monthly Long Distance Charges, dtype: int64

In [12]:
df["Customer Status"].value_counts(dropna=False)

Customer Status
Stayed     4720
Churned    1869
Joined      454
Name: count, dtype: int64

In [13]:
pd.crosstab(
    df["Customer Status"],
    df["Churn Category"].notna()
)

Churn Category,False,True
Customer Status,,
Churned,0,1869
Joined,454,0
Stayed,4720,0


In [14]:
pd.crosstab(
    df["Customer Status"],
    df["Churn Reason"].notna()
)

Churn Reason,False,True
Customer Status,,
Churned,0,1869
Joined,454,0
Stayed,4720,0


### Summary of the missing values investigation

The initial missing-value analysis identified three major structural patterns:

#### Internet service fields
- 1,526 customers have missing values across internet-service-related fields.
- All 1,526 customers have `Internet Service = No`.
- Customers with `Internet Service = Yes` have complete values for these fields.
- These missing values therefore appear to represent structurally inapplicable fields.

#### Phone service fields
- 682 customers have missing values for `Multiple Lines` and `Avg Monthly Long Distance Charges`.
- All 682 customers have `Phone Service = No`.
- Customers with `Phone Service = Yes` have complete values for these fields.
- These missing values therefore appear to represent structurally inapplicable fields.

#### Churn fields
- `Churn Category` and `Churn Reason` each contain 5,174 missing values.
- All 1,869 customers with `Customer Status = Churned` have both fields populated.
- All customers with `Customer Status = Stayed` or `Joined` have these fields missing.
- These values also appear to be structurally inapplicable rather than missing observations.

No records have been removed or modified during the initial audit.

In [15]:
df["Offer"].value_counts(dropna=False)

Offer
NaN        3877
Offer B     824
Offer E     805
Offer D     602
Offer A     520
Offer C     415
Name: count, dtype: int64

In [16]:
pd.crosstab(
    df["Offer"].isna(),
    df["Customer Status"],
    margins=True
)

Customer Status,Churned,Joined,Stayed,All
Offer,,,,
False,818,175,2173,3166
True,1051,279,2547,3877
All,1869,454,4720,7043


### Offer Field Investigation

The `Offer` field contains 3,877 missing values.

Missing `Offer` values occur across all three customer-status groups:
- Churned: 1,051
- Joined: 279
- Stayed: 2,547

Because missing values are not exclusive to a particular customer-status group, customer status alone does not explain the missingness.

At this stage, the reason for the missing `Offer` values is not established. The values will not be modified until additional investigation provides sufficient evidence for an appropriate treatment.

### Duplicates

In [17]:
df.duplicated().sum()

np.int64(0)

In [18]:
df["Customer ID"].duplicated().sum()

np.int64(0)

### Duplicate Check

The dataset contains no completely duplicated rows and no duplicate Customer IDs.

- Duplicate rows: 0
- Duplicate Customer IDs: 0

This indicates that each customer is represented by a unique record in the dataset.

In [19]:
df["Age"].describe()

count    7043.000000
mean       46.509726
std        16.750352
min        19.000000
25%        32.000000
50%        46.000000
75%        60.000000
max        80.000000
Name: Age, dtype: float64

In [20]:
df["Monthly Charge"].describe()

count    7043.000000
mean       63.596131
std        31.204743
min       -10.000000
25%        30.400000
50%        70.050000
75%        89.750000
max       118.750000
Name: Monthly Charge, dtype: float64

##### Investigating negative values for `Monthly Charge`

In [60]:
df[df["Monthly Charge"] < 0][
    ["Customer ID", "Monthly Charge", "Tenure in Months", "Total Charges"]
]

,Customer ID,Monthly Charge,Tenure in Months,Total Charges
1,0003-MKNFE,-4.0,9,542.40
32,0052-DCKON,-2.0,66,7942.15
170,0274-JKUJR,-3.0,7,465.70
232,0361-HJRDX,-8.0,68,4539.60
336,0495-ZBNGW,-1.0,5,343.95
...,...,...,...,...
6894,9793-WECQC,-1.0,23,2192.90
6918,9821-BESNZ,-7.0,66,6860.60
6976,9898-KZQDZ,-1.0,40,3640.45
7020,9961-JBNMK,-1.0,21,2030.30


In [24]:
df[df["Monthly Charge"] < 0][
    ["Customer ID", "Monthly Charge", "Tenure in Months", "Total Charges"]
].describe()

,Monthly Charge,Tenure in Months,Total Charges
count,120.000000,120.000000,120.000000
mean,-5.416667,29.766667,2028.856250
std,2.888934,24.643929,2120.436069
min,-10.000000,1.000000,19.400000
25%,-8.000000,7.000000,343.025000
50%,-5.000000,22.500000,1325.975000
75%,-3.000000,54.000000,3239.512500
max,-1.000000,72.000000,7942.150000


In [25]:
df[df["Monthly Charge"] < 0][
    ["Monthly Charge", "Offer", "Internet Service",
     "Internet Type", "Contract", "Payment Method"]
].value_counts().head(20)

Monthly Charge  Offer    Internet Service  Internet Type  Contract        Payment Method 
-7.0            Offer D  Yes               Fiber Optic    Month-to-Month  Bank Withdrawal    2
-10.0           Offer B  Yes               Cable          Two Year        Credit Card        1
-3.0            Offer B  Yes               Fiber Optic    Month-to-Month  Bank Withdrawal    1
-5.0            Offer D  Yes               Fiber Optic    One Year        Credit Card        1
                Offer E  Yes               Fiber Optic    Month-to-Month  Mailed Check       1
-4.0            Offer A  Yes               DSL            Two Year        Credit Card        1
                Offer E  Yes               Cable          Month-to-Month  Credit Card        1
                                           DSL            Month-to-Month  Bank Withdrawal    1
-3.0            Offer B  Yes               Fiber Optic    Two Year        Credit Card        1
-6.0            Offer B  Yes               DSL         

In [26]:
df[df["Monthly Charge"] < 0]["Monthly Charge"].value_counts().sort_index()


Monthly Charge
-10.0    13
-9.0      9
-8.0     12
-7.0     16
-6.0      6
-5.0     11
-4.0     17
-3.0     13
-2.0     10
-1.0     13
Name: count, dtype: int64

In [27]:
df.loc[df["Monthly Charge"] < 0, "Monthly Charge"].head(10).tolist()

[-4.0, -2.0, -3.0, -8.0, -1.0, -10.0, -2.0, -4.0, -7.0, -9.0]

### Monthly Charge Investigation

The `Monthly Charge` field contains 120 negative values.

The negative values range from -1 to -10 and occur across customers with different tenure lengths, customer statuses, contracts, and total charges.

The distribution is:

- -1: 13 customers
- -2: 10 customers
- -3: 13 customers
- -4: 17 customers
- -5: 11 customers
- -6: 6 customers
- -7: 16 customers
- -8: 12 customers
- -9: 9 customers
- -10: 13 customers

Because `Monthly Charge` represents a customer charge, negative values are suspicious. However, their underlying cause has not been established, so they will not be automatically converted or removed during the initial audit.

These records will be flagged for further investigation before final analysis.

In [28]:
df[[
    "Number of Dependents",
    "Number of Referrals",
    "Tenure in Months",
    "Avg Monthly GB Download",
    "Total Charges",
    "Total Refunds",
    "Total Extra Data Charges",
    "Total Long Distance Charges",
    "Total Revenue"
]].describe().T

,count,mean,std,min,25%,50%,75%,max
Number of Dependents,7043.0,0.468692,0.962802,0.00,0.000,0.00,0.000,9.00
Number of Referrals,7043.0,1.951867,3.001199,0.00,0.000,0.00,3.000,11.00
Tenure in Months,7043.0,32.386767,24.542061,1.00,9.000,29.00,55.000,72.00
Avg Monthly GB Download,5517.0,26.189958,19.586585,2.00,13.000,21.00,30.000,85.00
Total Charges,7043.0,2280.381264,2266.220462,18.80,400.150,1394.55,3786.600,8684.80
Total Refunds,7043.0,1.962182,7.902614,0.00,0.000,0.00,0.000,49.79
Total Extra Data Charges,7043.0,6.860713,25.104978,0.00,0.000,0.00,0.000,150.00
Total Long Distance Charges,7043.0,749.099262,846.660055,0.00,70.545,401.44,1191.100,3564.72
Total Revenue,7043.0,3034.379056,2865.204542,21.36,605.610,2108.64,4801.145,11979.34


### Numeric Field Range Check

The main numeric fields were reviewed for unusual ranges.

Most numeric variables have plausible ranges based on their definitions:

- Number of Dependents: 0–9
- Number of Referrals: 0–11
- Tenure in Months: 1–72
- Avg Monthly GB Download: 2–85
- Total Charges: 18.80–8,684.80
- Total Refunds: 0–49.79
- Total Extra Data Charges: 0–150
- Total Long Distance Charges: 0–3,564.72
- Total Revenue: 21.36–11,979.34

No obvious range problems were identified in these fields.

The main numeric data-quality issue identified so far is the presence of 120 negative `Monthly Charge` values, ranging from -1 to -10. These values have been documented for further consideration during the cleaning stage.

In [29]:
categorical_columns = df.select_dtypes(include="object").columns

for col in categorical_columns:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False))


--- Customer ID ---
Customer ID
0002-ORFBO    1
6616-AALSR    1
6625-UTXEW    1
6625-IUTTT    1
6625-FLENO    1
             ..
3352-RICWQ    1
3352-ALMCK    1
3351-NQLDI    1
3351-NGXYI    1
9995-HOTOH    1
Name: count, Length: 7043, dtype: int64

--- Gender ---
Gender
Male      3555
Female    3488
Name: count, dtype: int64

--- Married ---
Married
No     3641
Yes    3402
Name: count, dtype: int64

--- City ---
City
Los Angeles         293
San Diego           285
San Jose            112
Sacramento          108
San Francisco       104
                   ... 
Johannesburg          2
South Lake Tahoe      2
Jacumba               2
Holtville             2
Eldridge              2
Name: count, Length: 1106, dtype: int64

--- Offer ---
Offer
NaN        3877
Offer B     824
Offer E     805
Offer D     602
Offer A     520
Offer C     415
Name: count, dtype: int64

--- Phone Service ---
Phone Service
Yes    6361
No      682
Name: count, dtype: int64

--- Multiple Lines ---
Multiple Lines
No   